# Silver Metadata Helpers

Load per-table Silver pipeline configuration from JSON files committed in the repository.


In [ ]:
import json
from pathlib import Path

PIPELINE_CONFIG_SUBDIR = Path('pipeline_configs') / 'silver'

In [ ]:
def get_notebook_context_path() -> str | None:
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except Exception:
        return None


def candidate_repo_roots() -> list[Path]:
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    notebook_context_path = get_notebook_context_path()
    if notebook_context_path:
        notebook_path = Path(notebook_context_path)
        if notebook_path.is_absolute():
            candidates.extend([notebook_path.parent, *notebook_path.parents])

    candidates.extend([Path('/databricks/driver'), Path('/Workspace'), Path('/Workspace/Repos')])

    unique_candidates: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        unique_candidates.append(resolved)
    return unique_candidates


def resolve_pipeline_config_dir() -> Path:
    for root in candidate_repo_roots():
        candidate = root / PIPELINE_CONFIG_SUBDIR
        if candidate.is_dir():
            print(f'Loaded Silver pipeline configs from {candidate}')
            return candidate
    raise FileNotFoundError(f'Could not resolve pipeline config directory: {PIPELINE_CONFIG_SUBDIR}')


In [ ]:
def load_silver_table_configs() -> dict[str, dict]:
    config_dir = resolve_pipeline_config_dir()
    configs: dict[str, dict] = {}
    # Load from root dir and any subdirectories (e.g. dvdrental/)
    for config_path in sorted(config_dir.glob('**/*.json')):
        config = json.loads(config_path.read_text())
        if not config.get('enabled', True):
            continue
        table_id = config['table_id']
        configs[table_id] = config
    if not configs:
        raise ValueError(f'No enabled Silver pipeline configs found in {config_dir}')
    return configs


SILVER_TABLE_CONFIGS = load_silver_table_configs()


def list_enabled_silver_tables() -> list[str]:
    return sorted(SILVER_TABLE_CONFIGS.keys())


def get_silver_table_config(table_id: str) -> dict:
    if table_id not in SILVER_TABLE_CONFIGS:
        available = ', '.join(list_enabled_silver_tables())
        raise ValueError(f"Unknown TABLE_ID '{table_id}'. Available: {available}")
    return SILVER_TABLE_CONFIGS[table_id]

In [ ]:
def ensure_table_from_contract(spark, table_fqn: str, schema) -> None:
    if spark.catalog.tableExists(table_fqn):
        print(f"Using existing table: {table_fqn}")
        return

    empty_df = spark.createDataFrame([], schema)
    (
        empty_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_fqn)
    )
    print(f"Created table from contract: {table_fqn}")


In [ ]:
def run_silver_bootstrap_hook(spark, table_config: dict, silver_table_fqn: str, catalog: str, silver_schema: str) -> None:
    hook_name = table_config.get("bootstrap_hook")
    if not hook_name:
        return

    if hook_name == "reconcile_orders_legacy":
        products_table_fqn = f"{catalog}.{silver_schema}.silver_products"
        reconcile_silver_orders_table(spark, silver_table_fqn, products_table_fqn)
        return

    raise ValueError(f"Unsupported bootstrap hook: {hook_name}")
